# PY300 — Plotly — Interactive 2D and 3D Visualizations
### Official documentation: https://plotly.com/python/

**Plotly** creates rich, interactive charts that support hover, zoom, pan, and export — directly in Jupyter notebooks or as standalone HTML files.

## Why Plotly?
- **Interactive** — hover to see values, zoom into regions, toggle series on/off
- **Publication-ready** — beautiful defaults without CSS tuning
- **3D support** — scatter, surface, and mesh plots
- **Grammar of Graphics** — Plotly Express is built on this concept (more below)
- **Dash** — build full web dashboards using Plotly charts

## Grammar of Graphics (Concept)
Instead of calling `plot_bar()` or `plot_line()`, you **describe** a chart in layers:
1. **Data** — the DataFrame
2. **Aesthetics** — which columns map to x, y, colour, size
3. **Geometry** — bar, line, scatter, etc.
4. **Facets** — split into sub-plots by a category

Plotly Express (`px`) implements this — you specify *what* to show, not *how* to draw it.

> **Perspective:** Matplotlib is the foundation. Seaborn adds statistical aesthetics. Plotly adds interactivity. In a data science workflow, use Matplotlib/Seaborn for static reports (papers, PDFs) and Plotly for interactive dashboards and presentations.

In [ ]:
## Setup

import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np

---
## 2D Interactive Charts with Plotly Express

In [ ]:
## Interactive scatter plot — the Grammar of Graphics in action

# Using Plotly's built-in gapminder dataset
df = px.data.gapminder().query("year == 2007")

fig = px.scatter(
    df,
    x='gdpPercap',              # Aesthetic: x-axis
    y='lifeExp',                # Aesthetic: y-axis
    size='pop',                 # Aesthetic: bubble size
    color='continent',          # Aesthetic: colour grouping
    hover_name='country',       # Hover tooltip
    log_x=True,                 # Log scale for GDP
    size_max=60,
    title='GDP per Capita vs Life Expectancy (2007)'
)
fig.show()

# This single call maps 5 data dimensions to visual properties:
# x, y, size, color, hover — that is the Grammar of Graphics.

In [ ]:
## Interactive line chart with facets (small multiples)

df = px.data.gapminder().query("continent == 'Asia'")
top_countries = df.groupby('country')['pop'].max().nlargest(6).index
df_top = df[df['country'].isin(top_countries)]

fig = px.line(
    df_top,
    x='year', y='lifeExp',
    color='country',
    title='Life Expectancy Trend — Top 6 Asian Countries',
    markers=True
)
fig.show()

# Try: click legend items to toggle countries on/off, hover for exact values.

In [ ]:
## Interactive bar chart

df = px.data.gapminder().query("year == 2007 and continent == 'Europe'")
df = df.nlargest(15, 'gdpPercap')

fig = px.bar(
    df,
    x='country', y='gdpPercap',
    color='lifeExp',
    color_continuous_scale='Viridis',
    title='Top 15 European Countries by GDP per Capita (2007)',
    labels={'gdpPercap': 'GDP per Capita ($)', 'lifeExp': 'Life Expectancy'}
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [ ]:
## Animated scatter — GDP over time (press Play!)

df = px.data.gapminder()

fig = px.scatter(
    df,
    x='gdpPercap', y='lifeExp',
    animation_frame='year',       # Animation slider
    animation_group='country',
    size='pop', color='continent',
    hover_name='country',
    log_x=True, size_max=55,
    range_x=[100, 100000], range_y=[25, 90],
    title='World Development Over Time (Hans Rosling style)'
)
fig.show()

# This recreates the famous Hans Rosling "Gapminder" visualization.
# Interactivity and animation reveal patterns that static charts miss.

---
## 3D Interactive Charts

In [ ]:
## 3D Scatter Plot

df = px.data.iris()

fig = px.scatter_3d(
    df,
    x='sepal_length', y='sepal_width', z='petal_length',
    color='species',
    title='Iris Dataset — 3D Scatter',
    opacity=0.7
)
fig.show()

# Drag to rotate, scroll to zoom. 3D scatter helps visualize
# cluster separation that is hidden in 2D projections.

In [ ]:
## 3D Surface Plot

# Create a mathematical surface: z = sin(sqrt(x² + y²))
x = np.linspace(-5, 5, 50)
y = np.linspace(-5, 5, 50)
X, Y = np.meshgrid(x, y)
Z = np.sin(np.sqrt(X**2 + Y**2))

fig = go.Figure(data=[go.Surface(z=Z, x=X, y=Y, colorscale='Viridis')])
fig.update_layout(
    title='3D Surface — sin(√(x² + y²))',
    scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z')
)
fig.show()

---
## Plotly Graph Objects (`go`) — Full Control

While `px` (Plotly Express) is concise, `go` (Graph Objects) gives you low-level control over every trace, axis, and annotation.

In [ ]:
## Graph Objects — multi-trace chart with full control

months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun']
revenue = [100, 120, 115, 140, 155, 170]
cost = [80, 90, 95, 100, 105, 110]

fig = go.Figure()

fig.add_trace(go.Bar(x=months, y=revenue, name='Revenue', marker_color='#2196F3'))
fig.add_trace(go.Bar(x=months, y=cost, name='Cost', marker_color='#F44336'))
fig.add_trace(go.Scatter(
    x=months,
    y=[r - c for r, c in zip(revenue, cost)],
    name='Profit',
    mode='lines+markers',
    line=dict(color='#4CAF50', width=3),
    yaxis='y2'
))

fig.update_layout(
    title='Revenue, Cost & Profit',
    barmode='group',
    yaxis=dict(title='Amount (₹K)'),
    yaxis2=dict(title='Profit (₹K)', overlaying='y', side='right'),
    legend=dict(x=0.01, y=0.99)
)
fig.show()

---
## Major Plotly Features Summary

| Feature | Description |
|---|---|
| **Plotly Express (`px`)** | High-level, Grammar of Graphics-style API |
| **Graph Objects (`go`)** | Low-level, full control over every element |
| **Animations** | `animation_frame` parameter for time-based playback |
| **3D Charts** | `scatter_3d`, `surface`, `mesh3d`, `isosurface` |
| **Subplots** | `make_subplots()` for multi-panel layouts |
| **Facets** | `facet_col`, `facet_row` for small multiples |
| **Themes** | `template='plotly_dark'`, `'seaborn'`, etc. |
| **Export** | `.write_html()`, `.write_image()` (PNG/SVG/PDF) |
| **Dash** | Build full interactive web dashboards |

### Matplotlib vs Seaborn vs Plotly

| Library | Interactivity | Learning Curve | Best For |
|---|---|---|---|
| **Matplotlib** | Static | Low | Papers, reports, quick exploration |
| **Seaborn** | Static | Low | Statistical plots, beautiful defaults |
| **Plotly** | Interactive | Medium | Dashboards, presentations, 3D |
| **Bokeh** | Interactive | Medium | Streaming/real-time data |
| **Altair** | Interactive | Low | Declarative grammar of graphics |

> **Perspective:** The Grammar of Graphics (Leland Wilkinson, 1999) changed data visualization by separating *what* you want to show from *how* to render it. Plotly Express, Altair, and R's ggplot2 all implement this idea. Once you learn the grammar, switching between libraries becomes trivial.